# Part 9, notebook 2 of 5: pause, ask, resume

Companion to *Human in the Loop*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 09a | `09a_where_a_human_belongs` | Where a person belongs in the pipeline | no |
| 09b | `09b_pause_and_resume` | Pause, ask, resume: `interrupt()` | no |
| 09c | `09c_who_to_ask` | Which cases to send: the review budget | no (reads the saved Part 8 runs) |
| 09d | `09d_does_the_human_help` | Does the human step help, and what does it cost? | no (reads the saved Part 7 runs) |
| 09e | `09e_queue_and_failures` | The review queue and what goes wrong | one cell uses the model server |

**The picture.** A save game. You walk away from the console, the game keeps your exact position, and tomorrow you press Continue. `interrupt()` is "stop here and wait", the **checkpointer** is the save slot, and the `thread_id` is the ticket number you quote when you come back to the counter.

Part 6 paused *before* a node (`interrupt_before`). Here the pause is *inside* a node, so the node can hand the reviewer a question and receive an answer.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (only 09e calls it)
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
%matplotlib inline
import hitl_eval as ev, hitl_graph as h
from langgraph.checkpoint.memory import MemorySaver
from hitl_view import show_review

## The node, in full

Read this once. `interrupt(payload)` is the whole mechanism: the first time it runs, the graph stops and `payload` goes to the caller. After `Command(resume=answer)`, the same line returns `answer`.

In [2]:
import inspect
print(inspect.getsource(h.human_review_node))

def human_review_node(state: HitlState) -> dict:
    REPLAYS.append(state["image"])           # anything here runs AGAIN on resume: keep code before interrupt() side-effect free
    payload = review_payload(state)
    valid = set(payload["all_candidate_ids"])
    audit = []
    while True:
        raw = interrupt(payload)             # first pass: pauses here. After Command(resume=x): returns x
        try:
            d = parse_decision(raw, valid)
            break
        except DecisionError as exc:         # bad answer: do not crash, do not guess, ask again with the reason
            audit.append({"image": state["image"], "event": "answer rejected", "why": str(exc), "answer": raw})
            payload = {**payload, "previous_answer_rejected": str(exc)}
    before = [k["id"] for k in state["kept"]]
    if d.action == "approve":
        after = before
    elif d.action == "reject":
        after = []
    else:
        after = sorted(d.keep_ids)
    kept = [k for k in state["kept"] 

## Start a run and let it pause

In [3]:
g = h.build_hitl("band", "all_nonpass", checkpointer=MemorySaver())
for im in ev.TEST:
    r = h.start(g, im, thread_id := f"ticket-{im}")
    if r["status"] == "waiting":
        break
cfg = {"configurable": {"thread_id": thread_id}}
snap = g.get_state(cfg)
print("image:", im, "| ticket:", thread_id)
print("status:", r["status"], "| the graph is waiting at node:", snap.next)
print("pipeline verdict:", r["payload"]["pipeline_verdict"], "|", r["payload"]["why_a_human"])
print("candidates shown to the reviewer:", [(c["id"], c["pipeline"]) for c in r["payload"]["candidates"]])

image: test_00002 | ticket: ticket-test_00002
status: waiting | the graph is waiting at node: ('human_review',)
pipeline verdict: review | 2 confident defect(s) survived the rules
candidates shown to the reviewer: [(0, 'kept'), (1, 'kept'), (2, 'rejected by classifier')]


In [4]:
show_review(r["payload"])

## Your turn: answer it yourself

Look at the picture above. Edit `MY_DECISION` and run the cell. The three actions:

- `approve`: the boxes the pipeline kept are exactly the real defects.
- `reject`: none of them is real.
- `edit` with `keep_ids`: these ids are real (this also lets you **rescue** a dashed grey box the pipeline threw away).

The cell afterwards shows the exact answer in yellow outlines, so you can see whether you were right. A real reviewer would not have that.

In [5]:
MY_DECISION = {"action": "approve", "reviewer": "me", "note": ""}      # <- change me, e.g. {"action": "edit", "reviewer": "me", "keep_ids": [1]}

g2 = h.build_hitl("band", "all_nonpass", checkpointer=MemorySaver())     # a fresh graph, so you can rerun this cell
h.start(g2, im, "mine")
res = h.resume(g2, "mine", MY_DECISION)
if res["status"] == "waiting":
    print("the graph REFUSED your answer and asked again:", res["payload"]["previous_answer_rejected"])
else:
    s = res["state"]
    print("verdict:", s["verdict"], "| decided by:", s["decided_by"], "| kept:", [k["id"] for k in s["kept"]])
    t = ev.truth(im)
    print("the exact answer: real ids =", [i for i, v in t.items() if v])
    show_review(r["payload"], truth=t)

verdict: fail | decided by: human | kept: [0, 1]
the exact answer: real ids = [0]


## A wrong answer is asked again, never guessed

A reviewer uses a form, forms get filled in badly. The node validates every answer. If it is wrong it does **not** crash and does **not** pick a default: it asks again and says why. Every refused answer is kept in the audit trail.

In [6]:
g3 = h.build_hitl("band", "all_nonpass", checkpointer=MemorySaver())
h.start(g3, im, "bad")
bad = ["approve", {"action": "maybe", "reviewer": "x"}, {"action": "approve"}, {"action": "approve", "reviewer": "  "},
       {"action": "edit", "reviewer": "x"}, {"action": "edit", "reviewer": "x", "keep_ids": [999]},
       {"action": "approve", "reviewer": "x", "keep_ids": [0]}]
for b in bad:
    out = h.resume(g3, "bad", b)
    print(f"{str(b):62s} -> {out['status']}: {out['payload']['previous_answer_rejected']}")
final = h.resume(g3, "bad", {"action": "reject", "reviewer": "dee", "note": "dust"})
print("\nfinal:", final["state"]["verdict"], "|", final["state"]["reason"])
print("audit trail:")
for a in final["state"]["audit"]:
    print("  ", a["event"], "-", a.get("why") or {k: a[k] for k in ("reviewer", "action", "kept_before", "kept_after")})

approve                                                        -> waiting: expected a JSON object, got str
{'action': 'maybe', 'reviewer': 'x'}                           -> waiting: action: Input should be 'approve', 'reject' or 'edit'
{'action': 'approve'}                                          -> waiting: reviewer: Field required
{'action': 'approve', 'reviewer': '  '}                        -> waiting: reviewer must not be empty: every decision needs a name for the audit trail
{'action': 'edit', 'reviewer': 'x'}                            -> waiting: action 'edit' needs keep_ids (use [] to clear everything, or choose 'reject')
{'action': 'edit', 'reviewer': 'x', 'keep_ids': [999]}         -> waiting: keep_ids [999] are not candidates of this image (valid: [0, 1, 2])
{'action': 'approve', 'reviewer': 'x', 'keep_ids': [0]}        -> waiting: keep_ids is only allowed with action 'edit', not 'approve'

final: pass | dee chose 'reject': dust
audit trail:
   answer rejected - expected a

## The gotcha: the node starts again from its first line

When the reviewer answers, LangGraph does not continue from the middle of the node. It **runs the node again from the top**, and this time `interrupt()` returns the answer instead of pausing. Anything written before `interrupt()` runs twice. The counter below is exactly that: `REPLAYS` is appended to by the first line of the node.

Daily-life version: you phone the bank, get put on hold, and when the agent comes back they start the greeting again. If the greeting charged a fee, you paid twice.

In [7]:
h.REPLAYS.clear()
g4 = h.build_hitl("band", "all_nonpass", checkpointer=MemorySaver())
h.start(g4, im, "replay")
print("node body started", len(h.REPLAYS), "time(s) after the first run (it paused)")
h.resume(g4, "replay", {"action": "approve", "reviewer": "ana"})
print("node body started", len(h.REPLAYS), "time(s) after one answer")

node body started 1 time(s) after the first run (it paused)
node body started 2 time(s) after one answer


So the rule for code before `interrupt()`: it must be harmless to run again. In `human_review_node` the first line only appends to a demo list. Never send an email or charge a card before the `interrupt()`.

## The save slot survives a restart

`MemorySaver` lives in this Python process. A reviewer may answer tomorrow, after a deploy. `SqliteSaver` writes the state to a file. The next cell starts the run in this process and finishes it in a **brand-new Python process**.

In [8]:
import tempfile, subprocess, sys
from hitl_queue import ReviewQueue
db = tempfile.mktemp(suffix=".db")
q = ReviewQueue(db)
for im_ in ev.TEST[:12]:
    q.submit(im_)
tid = q.pending()[0]["thread_id"]
print("this process: queue has", q.counts(), "| oldest waiting:", tid)
code_ = (f"import sys; sys.path.insert(0, {str(Path('..').resolve())!r}); from hitl_queue import ReviewQueue; q = ReviewQueue({db!r}); "
         f"print('NEW process sees', q.counts()); r = q.answer({tid!r}, {{'action': 'approve', 'reviewer': 'ana'}}); "
         f"print('NEW process finished it:', r['state']['verdict'], r['state']['decided_by'])")
print(subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True).stdout)
print("this process again:", ReviewQueue(db).counts())

this process: queue has {'done': 8, 'waiting': 4} | oldest waiting: insp-test_00002


NEW process sees {'done': 8, 'waiting': 4}
NEW process finished it: fail human

this process again: {'done': 9, 'waiting': 3}
